# YOLOv8-cls Manual Quantization Tester

Validates the NumPy / C++ post-training quantization pipeline directly on the ONNX graph (no Keras / TensorFlow) using a RAM-safe subsample of `dataset/val/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

import numpy as np
import onnxruntime as ort

# Ensure project root is in python search path
sys.path.append(os.getcwd())

from quantization import QuantContainer, native
from quantization.evaluator import (evaluate_model, export_yolov8_cls,
                                    load_dataset)

print("C++ kernels:", "enabled" if native.available() else
      f"disabled ({native.load_error()})")

## Step 1: Get the ONNX model

Uses `yolov8n-cls.onnx` if it exists, otherwise exports it with `ultralytics` (`uv sync --extra export`).

In [ ]:
onnx_path = "yolov8n-cls.onnx"
if not os.path.exists(onnx_path):
    onnx_path = export_yolov8_cls("yolov8n-cls.pt", imgsz=224)
print(f"ONNX model: {onnx_path}")

## Step 2: Calibrate the Quantization Container

In [ ]:
quant_graph = QuantContainer(onnx_path, quant_type="int16")

if os.path.exists("dataset/val"):
    print("Loading real dataset images for calibration...")
    calib_loader = load_dataset("dataset/val", batch_size=4,
                                limit_per_class=2, limit_classes=1000)
    calib_data = [x_batch for x_batch, _ in calib_loader]
else:
    print("Dataset directory not found. Calibrating with dummy random inputs...")
    calib_data = [np.random.uniform(0.0, 1.0, (4, 3, 224, 224)).astype(np.float32)
                  for _ in range(3)]

quant_graph.calibrate(calib_data)
print("Calibration completed.")

## Step 3: Evaluate (ONNX Runtime vs NumPy float vs Quantized)

In [ ]:
ort_session = ort.InferenceSession(onnx_path)

evaluate_model(
    ort_session=ort_session,
    quant_graph=quant_graph,
    dataset_dir="dataset/val",
    batch_size=1,
    infer_mode="integer",
)

## Step 4: Layer-by-Layer Output Differences (float vs Quantized)

In [ ]:
val_loader = load_dataset("dataset/val", batch_size=2,
                          limit_per_class=2, limit_classes=1000)
x_sample, _ = next(val_loader)

layer_diffs = quant_graph.compare_layers(x_sample, mode="integer")

print(f"{'Layer Name':<45} | {'Layer Type':<20} | {'Mean Abs Diff':<15} | {'Max Abs Diff':<15}")
print("-" * 105)
for name, m in layer_diffs.items():
    print(f"{name:<45} | {m['layer_type']:<20} | {m['mean_diff']:<15.6f} | {m['max_diff']:<15.6f}")

## Step 5: Speed (C++ kernels vs NumPy fallback)

In [ ]:
import time

def bench(x, n=3):
    quant_graph.quantized_infer(x, mode="integer")  # warm-up
    t = time.perf_counter()
    for _ in range(n):
        quant_graph.quantized_infer(x, mode="integer")
    return (time.perf_counter() - t) / n

x = x_sample[:1]
fast = bench(x)
lib, native._lib = native._lib, None
try:
    slow = bench(x, n=1)
finally:
    native._lib = lib
print(f"C++ kernels : {fast * 1e3:8.1f} ms / image")
print(f"NumPy       : {slow * 1e3:8.1f} ms / image  ({slow / fast:.0f}x slower)")